# J/psi HDF5 smoke-test plots

This standalone notebook validates and plots a MadGraph J/psi HDF5 sample.

## Assumed event layout

The preferred dataset is:

```text
FDL/zData : shape (N, 8)
```

with columns:

```text
[lepton- px, lepton- py, lepton- pz, lepton- E,
 lepton+ px, lepton+ py, lepton+ pz, lepton+ E]
```

The notebook and HDF5 file are assumed to be in the **same folder**. Run the notebook from top to bottom before starting OTUS training.

In [ ]:
# =========================
# 1. Imports
# =========================

%matplotlib inline

from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.figsize": (8, 5.5),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print("Notebook working directory:", Path.cwd().resolve())

## 2. User settings

Leave `HDF5_FILENAME = None` to auto-detect the file when exactly one `.hdf5` or `.h5` file is present beside the notebook.

When multiple HDF5 files are present, set the filename explicitly.

In [ ]:
# =========================
# 2. User settings
# =========================

# Example:
# HDF5_FILENAME = "jpsi_smoke_test.hdf5"
HDF5_FILENAME = None

# Preferred HDF5 dataset. The loader tries alternatives if this is absent.
PREFERRED_DATASET = "FDL/zData"

# Physical J/psi reference mass in GeV.
JPSI_MASS = 3.0969

# Broad mass plot.
MASS_RANGE_BROAD = (2.5, 3.7)
MASS_BINS_BROAD = 120

# Fine mass plot around the J/psi reference mass.
MASS_ZOOM_HALF_WIDTH = 0.05
MASS_BINS_ZOOM = 160

# Suppress only extreme display tails in kinematic plots.
DISPLAY_PERCENTILE = 99.5

# Save figures beside the notebook.
SAVE_FIGURES = True
OUTPUT_DIR = Path("jpsi_smoke_test_plots")

In [ ]:
# =========================
# 3. Locate the HDF5 file
# =========================

def locate_hdf5_file(filename=None):
    if filename is not None:
        path = Path(filename)
        if not path.exists():
            raise FileNotFoundError(
                f"Cannot find HDF5 file: {path.resolve()}\n"
                "Place it beside this notebook or correct HDF5_FILENAME."
            )
        return path

    candidates = sorted(
        list(Path(".").glob("*.hdf5")) + list(Path(".").glob("*.h5"))
    )

    if len(candidates) == 0:
        raise FileNotFoundError(
            "No .hdf5 or .h5 file was found in:\n"
            f"{Path.cwd().resolve()}"
        )

    if len(candidates) > 1:
        names = "\n".join(f"  - {p.name}" for p in candidates)
        raise RuntimeError(
            "More than one HDF5 file was found. Set HDF5_FILENAME explicitly.\n"
            f"Available files:\n{names}"
        )

    return candidates[0]


HDF5_FILE = locate_hdf5_file(HDF5_FILENAME)

if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using HDF5 file:", HDF5_FILE.resolve())
print("Figure output directory:", OUTPUT_DIR.resolve())

## 4. Inspect the HDF5 structure

This cell prints every group and dataset, including shape and data type. Check this output before trusting the plots.

In [ ]:
# =========================
# 4. Inspect HDF5 contents
# =========================

def inspect_hdf5(path):
    datasets = []

    with h5py.File(path, "r") as handle:
        print("Top-level keys:", list(handle.keys()))
        print()

        def visitor(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"Group:   {name}")
            elif isinstance(obj, h5py.Dataset):
                print(
                    f"Dataset: {name:<35} "
                    f"shape={str(obj.shape):<18} dtype={obj.dtype}"
                )
                datasets.append(name)

        handle.visititems(visitor)

    return datasets


AVAILABLE_DATASETS = inspect_hdf5(HDF5_FILE)

## 5. Load the dilepton four-vectors

The loader first tries `FDL/zData`, followed by several common names. It accepts `(N, 8)` and automatically transposes an `(8, N)` array.

In [ ]:
# =========================
# 5. Load and validate events
# =========================

def choose_dataset(handle, preferred="FDL/zData"):
    candidates = [
        preferred,
        "zData",
        "FDL/data",
        "events",
        "data",
        "z",
    ]

    for key in candidates:
        if key in handle and isinstance(handle[key], h5py.Dataset):
            return key

    all_datasets = []

    def collect(name, obj):
        if isinstance(obj, h5py.Dataset):
            all_datasets.append(name)

    handle.visititems(collect)

    compatible = []
    for key in all_datasets:
        shape = handle[key].shape
        if len(shape) == 2 and (shape[1] >= 8 or shape[0] == 8):
            compatible.append(key)

    if len(compatible) == 1:
        print(
            "Preferred dataset was not found. "
            f"Using the only compatible dataset: {compatible[0]}"
        )
        return compatible[0]

    raise KeyError(
        "Could not uniquely identify the event array.\n"
        f"Available datasets: {all_datasets}\n"
        "Set PREFERRED_DATASET to the correct dataset path."
    )


def load_dilepton_events(path, preferred_dataset="FDL/zData"):
    with h5py.File(path, "r") as handle:
        dataset_key = choose_dataset(handle, preferred_dataset)
        events = np.asarray(handle[dataset_key])

    print("Selected dataset:", dataset_key)
    print("Raw array shape:", events.shape)
    print("Raw data type:", events.dtype)

    if events.ndim != 2:
        raise ValueError(
            f"Expected a 2D event array, but got shape {events.shape}."
        )

    if events.shape[0] == 8 and events.shape[1] != 8:
        events = events.T
        print("Transposed event array to:", events.shape)

    if events.shape[1] < 8:
        raise ValueError(
            "The event array must contain at least eight columns:\n"
            "[l- px, l- py, l- pz, l- E, l+ px, l+ py, l+ pz, l+ E]\n"
            f"Loaded shape: {events.shape}"
        )

    if events.shape[1] > 8:
        print(
            f"Warning: the array has {events.shape[1]} columns. "
            "Only the first eight columns will be used."
        )

    events = np.asarray(events[:, :8], dtype=np.float64)

    finite = np.all(np.isfinite(events), axis=1)
    positive_energy = (events[:, 3] > 0.0) & (events[:, 7] > 0.0)
    keep = finite & positive_energy

    removed = len(events) - int(np.count_nonzero(keep))
    events = events[keep]

    if len(events) == 0:
        raise ValueError(
            "No valid events remain after finite-value and positive-energy checks."
        )

    print("Valid events:", len(events))
    print("Removed invalid events:", removed)
    print("First valid event:")
    print(events[0])

    return events, dataset_key


z_data, DATASET_KEY = load_dilepton_events(
    HDF5_FILE,
    preferred_dataset=PREFERRED_DATASET,
)

assert z_data.shape[1] == 8
assert np.all(np.isfinite(z_data))

## 6. Physics observables

The invariant mass is reconstructed directly from the stored four-vectors:

```text
m_ll^2 = (E_- + E_+)^2 - |p_- + p_+|^2
```

In [ ]:
# =========================
# 6. Physics observables
# =========================

def invariant_mass_from_p4(p4):
    # p4 columns: [px, py, pz, E]
    p4 = np.asarray(p4, dtype=np.float64)
    mass2 = p4[:, 3]**2 - np.sum(p4[:, :3]**2, axis=1)
    return np.sqrt(np.clip(mass2, 0.0, None))


def dilepton_invariant_mass(events):
    # events columns:
    # [l- px,py,pz,E, l+ px,py,pz,E]
    total_p4 = events[:, 0:4] + events[:, 4:8]
    return invariant_mass_from_p4(total_p4)


def transverse_momentum(px, py):
    return np.hypot(px, py)


def pseudorapidity(px, py, pz):
    pt = transverse_momentum(px, py)
    return np.arcsinh(
        np.divide(
            pz,
            pt,
            out=np.zeros_like(pz, dtype=np.float64),
            where=pt > 0,
        )
    )


def rapidity(px, py, pz, energy):
    numerator = energy + pz
    denominator = energy - pz
    valid = (numerator > 0.0) & (denominator > 0.0)

    result = np.full_like(energy, np.nan, dtype=np.float64)
    result[valid] = 0.5 * np.log(
        numerator[valid] / denominator[valid]
    )
    return result


p4_minus = z_data[:, 0:4]
p4_plus = z_data[:, 4:8]
p4_jpsi = p4_minus + p4_plus

m_ll = dilepton_invariant_mass(z_data)
m_minus = invariant_mass_from_p4(p4_minus)
m_plus = invariant_mass_from_p4(p4_plus)

pt_minus = transverse_momentum(p4_minus[:, 0], p4_minus[:, 1])
pt_plus = transverse_momentum(p4_plus[:, 0], p4_plus[:, 1])
pt_jpsi = transverse_momentum(p4_jpsi[:, 0], p4_jpsi[:, 1])

eta_minus = pseudorapidity(
    p4_minus[:, 0], p4_minus[:, 1], p4_minus[:, 2]
)
eta_plus = pseudorapidity(
    p4_plus[:, 0], p4_plus[:, 1], p4_plus[:, 2]
)
y_jpsi = rapidity(
    p4_jpsi[:, 0],
    p4_jpsi[:, 1],
    p4_jpsi[:, 2],
    p4_jpsi[:, 3],
)

print("Computed observables for", len(m_ll), "events.")

## 7. Numerical summary and smoke-test checks

A correct resonance sample should produce a dilepton mass close to `3.0969 GeV`. A very narrow parton-level peak is normal.

In [ ]:
# =========================
# 7. Numerical summary
# =========================

def print_summary(name, values, unit=""):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    suffix = f" {unit}" if unit else ""
    print(name)
    print(f"  entries : {len(values)}")
    print(f"  minimum : {np.min(values):.8g}{suffix}")
    print(f"  median  : {np.median(values):.8g}{suffix}")
    print(f"  mean    : {np.mean(values):.8g}{suffix}")
    print(f"  std     : {np.std(values):.8g}{suffix}")
    print(f"  maximum : {np.max(values):.8g}{suffix}")
    print()


print_summary("Dilepton invariant mass", m_ll, "GeV")
print_summary("Negative-lepton mass", m_minus, "GeV")
print_summary("Positive-lepton mass", m_plus, "GeV")
print_summary("J/psi transverse momentum", pt_jpsi, "GeV")

mass_offset = np.median(m_ll) - JPSI_MASS
fraction_broad = np.mean(
    (m_ll >= MASS_RANGE_BROAD[0]) & (m_ll <= MASS_RANGE_BROAD[1])
)
fraction_10mev = np.mean(np.abs(m_ll - JPSI_MASS) <= 0.010)
fraction_50mev = np.mean(np.abs(m_ll - JPSI_MASS) <= 0.050)

print(f"Median mass minus reference: {mass_offset:+.8f} GeV")
print(
    f"Fraction in {MASS_RANGE_BROAD[0]}-{MASS_RANGE_BROAD[1]} GeV: "
    f"{fraction_broad:.4%}"
)
print(f"Fraction within 10 MeV of J/psi: {fraction_10mev:.4%}")
print(f"Fraction within 50 MeV of J/psi: {fraction_50mev:.4%}")

if abs(mass_offset) > 0.1:
    print()
    print("WARNING:")
    print(
        "The reconstructed median mass is more than 0.1 GeV "
        "from the J/psi reference mass."
    )
    print(
        "Check the dataset key, units, and column ordering "
        "before using this file."
    )

## 8. Broad J/psi mass plot

In [ ]:
# =========================
# 8. Broad mass distribution
# =========================

fig, ax = plt.subplots(constrained_layout=True)

bins = np.linspace(
    MASS_RANGE_BROAD[0],
    MASS_RANGE_BROAD[1],
    MASS_BINS_BROAD + 1,
)

ax.hist(
    m_ll,
    bins=bins,
    histtype="step",
    linewidth=2,
    label="MG5 J/psi sample",
)
ax.axvline(
    JPSI_MASS,
    linestyle="--",
    linewidth=1.5,
    label=r"$m_{J/\psi}=3.0969$ GeV",
)

ax.set_xlim(*MASS_RANGE_BROAD)
ax.set_xlabel(r"$m_{\ell\ell}$ [GeV]")
ax.set_ylabel("Events")
ax.set_title(r"J/$\psi \rightarrow \ell^+\ell^-$ HDF5 smoke test")
ax.legend(frameon=False)

if SAVE_FIGURES:
    path = OUTPUT_DIR / "jpsi_mass_broad.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path.resolve())

plt.show()

## 9. Zoomed J/psi mass plot

This uses a fixed window around the known J/psi mass. For an on-shell MadGraph sample, the distribution can be much narrower than the displayed window.

In [ ]:
# =========================
# 9. Zoomed mass distribution
# =========================

zoom_min = JPSI_MASS - MASS_ZOOM_HALF_WIDTH
zoom_max = JPSI_MASS + MASS_ZOOM_HALF_WIDTH

fig, ax = plt.subplots(constrained_layout=True)

bins = np.linspace(zoom_min, zoom_max, MASS_BINS_ZOOM + 1)

ax.hist(
    m_ll,
    bins=bins,
    histtype="step",
    linewidth=2,
    label="MG5 J/psi sample",
)
ax.axvline(
    JPSI_MASS,
    linestyle="--",
    linewidth=1.5,
    label=r"$m_{J/\psi}=3.0969$ GeV",
)

ax.set_xlim(zoom_min, zoom_max)
ax.set_xlabel(r"$m_{\ell\ell}$ [GeV]")
ax.set_ylabel("Events")
ax.set_title(r"J/$\psi$ mass peak — zoomed")
ax.legend(frameon=False)

if SAVE_FIGURES:
    path = OUTPUT_DIR / "jpsi_mass_zoomed.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path.resolve())

plt.show()

## 10. Daughter-lepton transverse momentum

The positive- and negative-lepton spectra should have compatible shapes for a symmetric decay sample.

In [ ]:
# =========================
# 10. Daughter pT distributions
# =========================

combined_pt = np.concatenate([pt_minus, pt_plus])
finite_combined_pt = combined_pt[np.isfinite(combined_pt)]
pt_upper = np.percentile(finite_combined_pt, DISPLAY_PERCENTILE)

if not np.isfinite(pt_upper) or pt_upper <= 0:
    raise ValueError("Could not determine a valid pT plotting range.")

fig, ax = plt.subplots(constrained_layout=True)

bins = np.linspace(0.0, pt_upper, 81)

ax.hist(
    pt_minus,
    bins=bins,
    histtype="step",
    linewidth=2,
    density=True,
    label=r"$\ell^-$",
)
ax.hist(
    pt_plus,
    bins=bins,
    histtype="step",
    linewidth=2,
    density=True,
    label=r"$\ell^+$",
)

ax.set_xlim(0.0, pt_upper)
ax.set_xlabel(r"Lepton $p_T$ [GeV]")
ax.set_ylabel("Normalized events")
ax.set_title(r"J/$\psi$ daughter-lepton transverse momentum")
ax.legend(frameon=False)

if SAVE_FIGURES:
    path = OUTPUT_DIR / "jpsi_daughter_pt.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path.resolve())

plt.show()

## 11. J/psi transverse momentum

In [ ]:
# =========================
# 11. J/psi pT distribution
# =========================

finite_pt_jpsi = pt_jpsi[np.isfinite(pt_jpsi)]

if len(finite_pt_jpsi) == 0:
    raise ValueError("No finite J/psi pT values are available.")

pt_jpsi_upper = np.percentile(finite_pt_jpsi, DISPLAY_PERCENTILE)
zero_fraction = np.mean(np.isclose(finite_pt_jpsi, 0.0, atol=1e-12))

print(f"J/psi pT minimum: {np.min(finite_pt_jpsi):.8g} GeV")
print(f"J/psi pT median:  {np.median(finite_pt_jpsi):.8g} GeV")
print(f"J/psi pT maximum: {np.max(finite_pt_jpsi):.8g} GeV")
print(f"Fraction consistent with zero pT: {zero_fraction:.4%}")

fig, ax = plt.subplots(constrained_layout=True)

if not np.isfinite(pt_jpsi_upper):
    raise ValueError("Could not determine a finite J/psi pT plotting range.")

if pt_jpsi_upper <= 0.0 or np.allclose(finite_pt_jpsi, 0.0, atol=1e-12):
    # For a leading-order 2 -> 1 resonance process with no recoil,
    # the generated J/psi can have exactly zero transverse momentum.
    # Draw a narrow display bin around zero instead of failing.
    display_half_width = 0.05  # GeV
    bins = np.linspace(-display_half_width, display_half_width, 3)

    ax.hist(
        finite_pt_jpsi,
        bins=bins,
        histtype="step",
        linewidth=2,
    )
    ax.axvline(0.0, linestyle="--", linewidth=1.5)
    ax.set_xlim(-display_half_width, display_half_width)
    ax.set_xlabel(r"$p_T(J/\psi)$ [GeV]")
    ax.set_ylabel("Events")
    ax.set_title(
        r"Generated J/$\psi$ transverse momentum "
        "(all events at $p_T=0$)"
    )

    print(
        "All J/psi transverse momenta are zero. "
        "This is expected for a leading-order sample without a recoiling jet "
        "or initial-state radiation."
    )
else:
    bins = np.linspace(0.0, pt_jpsi_upper, 81)

    ax.hist(
        finite_pt_jpsi,
        bins=bins,
        histtype="step",
        linewidth=2,
    )

    ax.set_xlim(0.0, pt_jpsi_upper)
    ax.set_xlabel(r"$p_T(J/\psi)$ [GeV]")
    ax.set_ylabel("Events")
    ax.set_title(r"Generated J/$\psi$ transverse momentum")

if SAVE_FIGURES:
    path = OUTPUT_DIR / "jpsi_pt.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path.resolve())

plt.show()


## 12. Angular distributions

This is a quick check for corrupted momentum components or unit mistakes.

In [ ]:
# =========================
# 12. Eta and rapidity distributions
# =========================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.8),
    constrained_layout=True,
)

eta_finite = np.concatenate([
    eta_minus[np.isfinite(eta_minus)],
    eta_plus[np.isfinite(eta_plus)],
])
eta_limit = np.percentile(np.abs(eta_finite), DISPLAY_PERCENTILE)
eta_limit = max(float(eta_limit), 0.5)

eta_bins = np.linspace(-eta_limit, eta_limit, 81)

axes[0].hist(
    eta_minus,
    bins=eta_bins,
    histtype="step",
    linewidth=2,
    density=True,
    label=r"$\ell^-$",
)
axes[0].hist(
    eta_plus,
    bins=eta_bins,
    histtype="step",
    linewidth=2,
    density=True,
    label=r"$\ell^+$",
)
axes[0].set_xlim(-eta_limit, eta_limit)
axes[0].set_xlabel(r"Lepton $\eta$")
axes[0].set_ylabel("Normalized events")
axes[0].set_title("Daughter pseudorapidity")
axes[0].legend(frameon=False)

finite_y = y_jpsi[np.isfinite(y_jpsi)]
if len(finite_y) > 0:
    y_limit = np.percentile(np.abs(finite_y), DISPLAY_PERCENTILE)
    y_limit = max(float(y_limit), 0.5)
    y_bins = np.linspace(-y_limit, y_limit, 81)

    axes[1].hist(
        finite_y,
        bins=y_bins,
        histtype="step",
        linewidth=2,
    )
    axes[1].set_xlim(-y_limit, y_limit)

axes[1].set_xlabel(r"$y(J/\psi)$")
axes[1].set_ylabel("Events")
axes[1].set_title(r"J/$\psi$ rapidity")

if SAVE_FIGURES:
    path = OUTPUT_DIR / "jpsi_eta_rapidity.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path.resolve())

plt.show()

## 13. Final status

The sample passes the basic smoke test when:

- the array is two-dimensional with eight usable columns;
- all retained values are finite and both stored energies are positive;
- the dilepton invariant-mass peak is near `3.0969 GeV`;
- daughter and J/psi kinematic distributions are finite and physically plausible.

A peak far from `3.1 GeV` usually indicates a wrong dataset, wrong units, or wrong column ordering rather than a plotting issue.

In [ ]:
# =========================
# 13. Final smoke-test status
# =========================

mass_tolerance = 0.1  # GeV; deliberately loose for a smoke test
median_mass = float(np.median(m_ll))

checks = {
    "event array has shape (N, 8)": (
        z_data.ndim == 2 and z_data.shape[1] == 8
    ),
    "all retained event values are finite": bool(
        np.all(np.isfinite(z_data))
    ),
    "all retained lepton energies are positive": bool(
        np.all(z_data[:, 3] > 0.0)
        and np.all(z_data[:, 7] > 0.0)
    ),
    "median dilepton mass is near J/psi": (
        abs(median_mass - JPSI_MASS) <= mass_tolerance
    ),
    "J/psi pT values are finite": bool(
        np.all(np.isfinite(pt_jpsi))
    ),
}

for label, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {label}")

overall = all(checks.values())
print()
print(
    "OVERALL RESULT:",
    "PASS" if overall else "CHECK THE FILE BEFORE TRAINING",
)

if SAVE_FIGURES:
    print("Saved figures in:", OUTPUT_DIR.resolve())